In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import distinctipy
import pickle
from collections import Counter
from tqdm import tqdm 
import cv2
from pathlib import Path
from scipy.ndimage import binary_dilation
!pip install joblib
from joblib import Parallel, delayed
import napari

In [ ]:
#given labeled image and cell_segmentation_mask and list of cell ids
#for each cell compute the proportion of 0, 1, 2, 3, 4 in window (radius of 100 pixels )
#filter out cells where < 90% of window is 1, 2, 3 
#project 1, 2, 3 for each cell into barycentric coordinates 

In [ ]:
import distinctipy
def create_color_dict(labels_unique, excluded_colors=None):
    color_arr = distinctipy.get_colors(len(labels_unique), exclude_colors=excluded_colors)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
radius = 50

In [ ]:
key = '/Multiplex 2/ANCA 9'

In [ ]:
cell_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering' + key

In [ ]:
fibrosis_path = 'Y:/coskun-lab/Aditi/01/Results/fibrosis_segmentation/' + key 

In [ ]:
cell_pixel_barcode_path = 'Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key

In [ ]:
registration_path = 'Y:/coskun-lab/Aditi/01/Results/registration/' + key + '/1x'

In [ ]:
with open(cell_clustering_base_path + '/1x/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_base_path + '/1x/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0', 'index', 'index.1'])

In [ ]:
with open(cell_clustering_base_path + '/1x/cell_type_color_dict.pickle', 'rb') as f: 
    cell_type_color_dict = pickle.load(f)

In [ ]:
with open(fibrosis_path + '/wsi_labeled.pickle', 'rb') as f: 
    fibrosis_segmentation_mask = pickle.load(f) 

In [ ]:
with open(fibrosis_path + '/cluster_color_dict.pickle', 'rb') as f: 
    fibrosis_cluster_color_dict = pickle.load(f) 

In [ ]:
num_fibrosis_clusters = np.nanmax(fibrosis_segmentation_mask)

In [ ]:
fibrosis_segmentation_mask_nan_removed = fibrosis_segmentation_mask.copy()
fibrosis_segmentation_mask_nan_removed[np.isnan(fibrosis_segmentation_mask_nan_removed)] = 0

In [ ]:
fibrosis_segmentation_mask_nan_removed = np.pad(fibrosis_segmentation_mask_nan_removed, ((0, cell_segmentation_mask.shape[0]-fibrosis_segmentation_mask_nan_removed.shape[0]), (0, cell_segmentation_mask.shape[1]-fibrosis_segmentation_mask_nan_removed.shape[1])))

In [ ]:
from scipy.ndimage import distance_transform_edt
def compute_pixel_count_outside_cell(cell_idx):
    ys, xs = np.where(cell_segmentation_mask == cell_idx)
    y_min = max(ys.min() - radius, 0)
    y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
    x_min = max(xs.min() - radius, 0)
    x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])
    
    cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_idx)
    label_block = fibrosis_segmentation_mask_nan_removed[y_min:y_max, x_min:x_max]
    dist = distance_transform_edt(~cell_block) #, metric='chessboard')
    ring_mask = (dist <= radius) & (~cell_block)

    pixels_outside = label_block[ring_mask]

    pixels_outside_int = pixels_outside.astype(np.uint8)
    counts = np.bincount(pixels_outside_int, minlength=int(num_fibrosis_clusters)+1)[1:]
    
    return {i+1: v for i, v in enumerate(counts)}


In [ ]:
cell_id_list = list(cell_cluster_location_df['Cell ID'])

In [ ]:
!pip install tqdm_joblib 
from tqdm_joblib import tqdm_joblib

In [ ]:
with tqdm_joblib(tqdm(desc="Processing cells", total=len(cell_id_list))) as progress_bar:
    pixel_counts_around_cells = Parallel(n_jobs=-1)(delayed(compute_pixel_count_outside_cell)(idx) for idx in cell_id_list)

In [ ]:
with open(fibrosis_path + '/pixel_counts_outside_cells.pickle', 'wb') as f: 
    pickle.dump(pixel_counts_around_cells, f)

In [ ]:
pixel_counts_around_cells_df = pd.DataFrame(pixel_counts_around_cells, columns=range(1, int(num_fibrosis_clusters)+1))
pixel_proportions_around_cells_df = pixel_counts_around_cells_df.copy()
pixel_proportions_around_cells_df = pixel_proportions_around_cells_df.div(pixel_counts_around_cells_df.sum(axis=1), axis=0)
cell_cluster_location_classified_df = cell_cluster_location_df[cell_cluster_location_df['Cell Type'] !='Unclassified']
pixel_proportions_around_cells_df['Cell ID'] = cell_cluster_location_classified_df['Cell ID']
pixel_proportions_around_cells_df['X'] = cell_cluster_location_classified_df['Y']
pixel_proportions_around_cells_df['Y'] = cell_cluster_location_classified_df['X']
pixel_proportions_around_cells_df['Cell Type'] = cell_cluster_location_classified_df['Cell Type']

In [ ]:
pixel_proportions_around_cells_na_removed_df = pixel_proportions_around_cells_df.dropna()

In [ ]:
pixel_proportions_around_cells_na_removed_processed_df = pixel_proportions_around_cells_na_removed_df.loc[pixel_proportions_around_cells_na_removed_df.loc[:,[1, 2, 3]].sum(axis=1)>0.9, :]

In [ ]:
pixel_proportions_around_cells_na_removed_processed_df = pixel_proportions_around_cells_na_removed_processed_df[pixel_proportions_around_cells_na_removed_processed_df['Cell Type']!='Unclassified']

In [ ]:
with open(registration_path + '/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
wgc  = pixel_proportions_around_cells_na_removed_processed_df
idx = wgc.index.values
x = wgc.loc[:,[1, 2, 3]]
proj = np.array([[0,0],[np.cos(np.pi/3),np.sin(np.pi/3)], [1,0]])
coords = np.dot(x,proj)

In [ ]:
cell_ids = cell_segmentation_mask.ravel()
fibrosis_types = fibrosis_segmentation_mask_nan_removed.ravel()

df_pixels = pd.DataFrame({
    "Cell ID": cell_ids,
    'Fibrosis Type': fibrosis_types
})
df_pixels = df_pixels[df_pixels['Cell ID']!=0].reset_index()

unique_values_by_cell = (
    df_pixels.groupby("Cell ID")['Fibrosis Type'].agg(lambda x: x.value_counts().idxmax())
)

cell_cluster_location_df['Fibrosis Type'] = cell_cluster_location_df['Cell ID'].map(unique_values_by_cell)

In [ ]:
cell_cluster_location_df = cell_cluster_location_df[cell_cluster_location_df["Cell ID"].isin(pixel_proportions_around_cells_na_removed_processed_df["Cell ID"])]

In [ ]:
fibrosis_cluster_color_dict[0] = (0.5, 0.5, 0.5)

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1)
i = 0

patches = [
    Line2D([], [], 
           color=color, 
           marker='o', 
           linestyle='None',
           markersize=12,
           label=label)
    for label, color in fibrosis_cluster_color_dict.items()
]

ax.legend(
    patches,
    list(fibrosis_cluster_color_dict.keys()),
    loc='center',
    frameon=False,
    ncol=len(fibrosis_cluster_color_dict)/len(fibrosis_cluster_color_dict), labelspacing=1
)

ax.axis('off')


In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1)
i = 0

patches = [
    Line2D([], [], 
           color=color, 
           marker='o', 
           linestyle='None',
           markersize=12,
           label=label)
    for label, color in cell_type_color_dict.items()
]

ax.legend(
    patches,
    list(cell_type_color_dict.keys()),
    loc='center',
    frameon=False,
    ncol=len(cell_type_color_dict)/len(cell_type_color_dict), labelspacing=1
)

ax.axis('off')


In [ ]:

target_cell_types = ['Epithelial cells', 'B-cells']
colors = []
for t in list(wgc['Cell Type']):
    if(t in target_cell_types):
        colors.append(cell_type_color_dict[t])
    else:
        colors.append((1, 1, 1, 0))
sizes = [4 for t in list(wgc['Cell Type'])]

In [ ]:
jit = 0.002
fig, ax = plt.subplots(1)
fig.set_figheight(8)
fig.set_figwidth(8)
ax.scatter(coords[:,0]+jit*np.random.randn(len(coords)),coords[:,1]+jit*np.random.randn(len(coords)), s = sizes, alpha = .5, c = colors) 
ax.scatter(x=[0, 1, np.cos(np.pi/3)], y=[0, 0, np.sin(np.pi/3)], c=[(0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0)], s=100)
ax.axis('off')

In [ ]:
del fibrosis_cluster_color_dict[0]

#### Visualize a selected cell as pie chart and the fibrosis subtype annotated image for the region containing the cell

In [ ]:
radius = 100
y, x = np.ogrid[-radius:radius+1, -radius:radius+1]
structure = (x**2 + y**2) <= radius**2

cell_id = 93245

pixel_proportions_around_cells_na_removed_processed_reset_df = pixel_proportions_around_cells_na_removed_processed_df.reset_index()

In [ ]:
pixel_proportions_around_cells_na_removed_processed_reset_df[pixel_proportions_around_cells_na_removed_processed_reset_df['Cell ID'] == 93245]

In [ ]:

ys, xs = np.where(cell_segmentation_mask == cell_id)
y_min = max(ys.min() - radius, 0)
y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
x_min = max(xs.min() - radius, 0)
x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])

cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_id)
label_block = fibrosis_segmentation_mask[y_min:y_max, x_min:x_max]


fig, ax = plt.subplots(1, 2)
fig.set_figheight(15)
fig.set_figwidth(15)
offset = 200
ax[0].imshow(cell_block)
ax[1].imshow(label_block, cmap=plt.cm.colors.ListedColormap(list(fibrosis_cluster_color_dict.values())), vmin=1, vmax=4)

In [ ]:
fig, ax = plt.subplots(1)
fig.set_figwidth(2)    
fig.set_figheight(2)

scale = 4 

i = 0  

barcode = np.array(pixel_proportions_around_cells_na_removed_processed_df[pixel_proportions_around_cells_na_removed_processed_df['Cell ID'] == cell_id])[0][:4]

sizes = barcode
colors = [fibrosis_cluster_color_dict[idx+1] for idx in range(len(barcode))]


pie_width = 0.15 * scale
pie_height = 0.15 * scale


pie_ax = ax.inset_axes([
    0, 0, 
    pie_width,
    pie_height
])

pie_ax.pie(
    sizes,
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'black'}
)
pie_ax.set_aspect('equal')
pie_ax.axis('off')

ax.axis('off')



#### Visualize each cell as pie chart in an ROI

In [ ]:
coord = roi['C']

pixel_proportions_around_cells_na_removed_subset_df = pixel_proportions_around_cells_na_removed_df[(pixel_proportions_around_cells_na_removed_df['X'].ge(coord[0])) & (pixel_proportions_around_cells_na_removed_df['X'].le(coord[1])) & (pixel_proportions_around_cells_na_removed_df['Y'].ge(coord[2])) & (pixel_proportions_around_cells_na_removed_df['Y'].le(coord[3]))]
target_X = pixel_proportions_around_cells_na_removed_subset_df[pixel_proportions_around_cells_na_removed_subset_df['Cell ID'] == cell_id]['X'].item()
target_Y = pixel_proportions_around_cells_na_removed_subset_df[pixel_proportions_around_cells_na_removed_subset_df['Cell ID'] == cell_id]['Y'].item()

colors = [
    fibrosis_cluster_color_dict[1],
    fibrosis_cluster_color_dict[2],
    fibrosis_cluster_color_dict[3],
    fibrosis_cluster_color_dict[4],
]

chart_size = 24
oversample = 8
def create_cell_as_chart(
    values,
    colors,
    size,
    oversample,
    border_color=(0.5, 0.5, 0.5),
    border_thickness=0.5
):
    H = W = size * oversample
    y, x = np.ogrid[:H, :W]
    cx = cy = (H - 1) / 2

    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    mask = r <= cx

    angles = (np.arctan2(y - cy, x - cx) + np.pi) / (2 * np.pi)
    cum = np.cumsum(values) / np.sum(values)

    img = np.zeros((H, W, 4), dtype=np.float32)

    start = 0.0
    for frac, color in zip(cum, colors):
        sector = (angles >= start) & (angles < frac) & mask
        img[sector, :3] = color
        img[sector, 3] = 1.0
        start = frac

    border_px = border_thickness * oversample
    border_mask = (r <= cx) & (r >= cx - border_px)

    img[border_mask, :3] = border_color
    img[border_mask, 3] = 1.0

    img = img.reshape(size, oversample, size, oversample, 4).mean(axis=(1, 3))
    return img



H = fibrosis_segmentation_mask.shape[0]
W = fibrosis_segmentation_mask.shape[1]

canvas = np.zeros((H, W, 4), dtype=np.float32)
cache = {}

half = chart_size // 2

for _, row in pixel_proportions_around_cells_na_removed_subset_df.iterrows():
    values = tuple(row[[1, 2, 3, 4]])

    y = int(row["X"])
    x = int(row["Y"])

    if(row['Cell ID'] == cell_id):
        print('target cell found')
        chart = create_cell_as_chart(values, colors, chart_size, oversample, border_color=(1, 0, 0), border_thickness=2)
    elif(values not in cache):
        cache[values] = create_cell_as_chart(values, colors, chart_size, oversample)
        chart = cache[values]
    else:
        chart = cache[values]

    

    y0 = max(0, y - half)
    y1 = min(H, y + half)
    x0 = max(0, x - half)
    x1 = min(W, x + half)

    cy0 = half - (y - y0)
    cy1 = cy0 + (y1 - y0)
    cx0 = half - (x - x0)
    cx1 = cx0 + (x1 - x0)

    dst = canvas[y0:y1, x0:x1]
    src = chart[cy0:cy1, cx0:cx1]

    alpha = src[..., 3:4]

    dst[..., :3] = src[..., :3] * alpha + dst[..., :3] * (1 - alpha)
    dst[..., 3] = alpha[..., 0] + dst[..., 3] * (1 - alpha[..., 0])

    canvas[y0:y1, x0:x1] = dst


viewer = napari.Viewer()
viewer.theme = "light"
viewer.add_image(
    canvas,
    rgb=True,
    interpolation2d="linear"
)
